# 4. 컬러 이미지 데이터셋에서 CNN 동작 보기

이 노트북은 `03_CNN_성능_개선.ipynb` 다음 단계로, 실제 컬러 이미지 데이터셋인 **CIFAR-10**에서 CNN이 어떻게 동작하는지 확인하는 실습입니다.

MNIST는 `(1, 28, 28)` 형태의 흑백 숫자 이미지였지만, CIFAR-10은 `(3, 32, 32)` 형태의 **RGB 컬러 이미지**입니다. 즉, CNN은 이제 하나의 채널이 아니라 `R`, `G`, `B` 세 채널을 함께 보면서 특징을 학습해야 합니다.

> 처음 실행할 때는 CIFAR-10 다운로드가 필요합니다. 네트워크 환경에 따라 시간이 조금 걸릴 수 있습니다.

## 목차

- [4-1. CIFAR-10 데이터셋 이해](#4-1.-CIFAR-10-데이터셋-이해)
- [4-2. 데이터 준비](#4-2.-데이터-준비)
- [4-3. 컬러 이미지용 CNN 정의](#4-3.-컬러-이미지용-CNN-정의)
- [4-4. 모델 학습](#4-4.-모델-학습)
- [4-5. 예측 결과 확인](#4-5.-예측-결과-확인)
- [4-6. 첫 번째 합성곱 필터 시각화](#4-6.-첫-번째-합성곱-필터-시각화)


## 4-1. CIFAR-10 데이터셋 이해

CIFAR-10은 총 10개의 클래스로 이루어진 대표적인 컬러 이미지 분류 데이터셋입니다.

- 클래스: `airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck`
- 학습 데이터: 50,000장
- 테스트 데이터: 10,000장
- 이미지 크기: `32 x 32`
- 채널 수: `3` (RGB)

여기서 중요한 점은, CNN의 첫 번째 합성곱 층이 **각 필터마다 3개 채널을 동시에 받아서 하나의 특징 맵을 만든다**는 것입니다. 즉, 필터는 단순히 밝기만 보는 것이 아니라 색 조합과 경계, 질감까지 함께 학습합니다.


In [ ]:
# 필요 라이브러리가 없다면 아래 주석을 해제해서 설치하세요.
# !pip install torch torchvision matplotlib


In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

torch.manual_seed(42)


## 4-2. 데이터 준비

컬러 이미지에서는 데이터 증강이 더 유용하게 작동하는 경우가 많습니다. 예를 들어 좌우 반전, 약간의 crop만으로도 모델이 더 다양한 모습을 보게 할 수 있습니다.

또한 CIFAR-10은 RGB 3채널이므로, 정규화도 채널별 평균과 표준편차를 사용합니다.


In [ ]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

full_train_aug = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
full_train_eval = datasets.CIFAR10(root='./data', train=True, download=False, transform=eval_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=eval_transform)

classes = full_train_aug.classes
train_size = 45000
val_size = 5000

generator = torch.Generator().manual_seed(42)
train_dataset, _ = random_split(full_train_aug, [train_size, val_size], generator=generator)
_, val_dataset = random_split(full_train_eval, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print('Classes:', classes)
print('Train samples:', len(train_dataset))
print('Validation samples:', len(val_dataset))
print('Test samples:', len(test_dataset))


In [ ]:
def denormalize(image):
    mean_tensor = torch.tensor(mean).view(3, 1, 1)
    std_tensor = torch.tensor(std).view(3, 1, 1)
    return (image.cpu() * std_tensor + mean_tensor).clamp(0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    image = denormalize(images[i]).permute(1, 2, 0)
    ax.imshow(image)
    ax.set_title(classes[labels[i].item()])
    ax.axis('off')

plt.suptitle('CIFAR-10 sample images')
plt.tight_layout()
plt.show()


위 이미지를 보면 CIFAR-10은 MNIST보다 훨씬 복잡합니다. 숫자 한 개가 화면 중앙에 있는 것이 아니라, 배경과 색상, 다양한 물체 모양이 함께 들어 있습니다.

따라서 CNN도 더 많은 채널과 더 깊은 층을 사용해 특징을 추출하는 것이 유리합니다.


## 4-3. 컬러 이미지용 CNN 정의

이번 모델은 입력 채널을 `1`이 아니라 `3`으로 받습니다. 첫 번째 층 `nn.Conv2d(3, 32, ...)`가 바로 RGB 이미지를 처리하는 부분입니다.

MNIST에서는 흑백 한 장을 보던 CNN이, 이제는 `R`, `G`, `B` 세 채널을 동시에 보면서 색상과 윤곽, 질감을 함께 학습합니다.

아래에서는 먼저 전체 구조를 순서도로 보고, 그 다음 각 계층을 자세히 설명합니다.

### 4-3-1. CNN 구조 순서도


In [ ]:
from html import escape
from IPython.display import SVG, display

def multiline_svg_text(x, y, lines, line_height=20, size=16, color='#111827', weight='500'):
    text = []
    for i, line in enumerate(lines):
        dy = i * line_height
        text.append(
            f"<text x='{x}' y='{y + dy}' text-anchor='middle' "
            f"font-size='{size}' font-weight='{weight}' fill='{color}' font-family='Segoe UI, Arial, sans-serif'>{escape(line)}</text>"
        )
    return ''.join(text)

blocks = [
    {'title': ['Input Image', '(3, 32, 32)'], 'subtitle': ['RGB channels'], 'fill': '#E0F2FE'},
    {'title': ['Block 1'], 'subtitle': ['Conv 3->32', 'BatchNorm', 'ReLU', 'Conv 32->32', 'ReLU', 'MaxPool -> (32, 16, 16)'], 'fill': '#FFEDD5'},
    {'title': ['Block 2'], 'subtitle': ['Conv 32->64', 'BatchNorm', 'ReLU', 'Conv 64->64', 'ReLU', 'MaxPool -> (64, 8, 8)'], 'fill': '#FEF3C7'},
    {'title': ['Block 3'], 'subtitle': ['Conv 64->128', 'BatchNorm', 'ReLU', 'MaxPool -> (128, 4, 4)'], 'fill': '#FCE7F3'},
    {'title': ['Classifier'], 'subtitle': ['Flatten -> 2048', 'Linear 2048->256', 'ReLU', 'Dropout 0.3', 'Linear 256->10'], 'fill': '#DCFCE7'},
    {'title': ['Output'], 'subtitle': ['10 class logits'], 'fill': '#E5E7EB'}
]

width = 1180
height = 330
box_w = 160
box_h = 168
y = 82
start_x = 30
gap = 24

svg = [
    f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
    "<defs>",
    "  <filter id='shadow' x='-20%' y='-20%' width='140%' height='140%'>",
    "    <feDropShadow dx='0' dy='6' stdDeviation='8' flood-color='#0F172A' flood-opacity='0.12' />",
    "  </filter>",
    "  <marker id='arrow' markerWidth='10' markerHeight='10' refX='9' refY='5' orient='auto'>",
    "    <path d='M 0 0 L 10 5 L 0 10 z' fill='#475569' />",
    "  </marker>",
    "</defs>",
    "<rect width='100%' height='100%' rx='24' fill='#FAFAF9' />",
    "<text x='590' y='38' text-anchor='middle' font-size='24' font-weight='700' fill='#111827' font-family='Segoe UI, Arial, sans-serif'>ColorCNN Architecture Flow</text>",
    "<text x='590' y='62' text-anchor='middle' font-size='13' fill='#475569' font-family='Segoe UI, Arial, sans-serif'>CIFAR-10 RGB image -> convolution blocks -> classifier -> 10 classes</text>"
]

for i, block in enumerate(blocks):
    x = start_x + i * (box_w + gap)
    svg.append(f"<rect x='{x}' y='{y}' width='{box_w}' height='{box_h}' rx='20' fill='{block['fill']}' stroke='#CBD5E1' stroke-width='1.5' filter='url(#shadow)' />")
    svg.append(multiline_svg_text(x + box_w / 2, y + 34, block['title'], line_height=20, size=18, weight='700'))
    svg.append(multiline_svg_text(x + box_w / 2, y + 78, block['subtitle'], line_height=18, size=13, color='#334155', weight='500'))
    if i < len(blocks) - 1:
        x1 = x + box_w
        x2 = x + box_w + gap - 6
        yc = y + box_h / 2
        svg.append(f"<line x1='{x1}' y1='{yc}' x2='{x2}' y2='{yc}' stroke='#475569' stroke-width='3' marker-end='url(#arrow)' />")

svg.append("</svg>")
display(SVG(''.join(svg)))


### 4-3-2. 각 계층이 하는 일 자세히 보기

이 모델은 입력 이미지가 분류 결과로 바뀔 때까지 여러 단계를 거칩니다. 핵심은 **앞쪽 층은 단순한 시각 특징을 보고, 뒤쪽 층은 그 특징들을 조합해 더 복잡한 물체 개념으로 바꾼다**는 점입니다.

#### 1) 입력층: `(3, 32, 32)`

- CIFAR-10 한 장은 `RGB` 3채널 이미지입니다.
- 텐서 shape는 `(채널 수, 높이, 너비) = (3, 32, 32)`입니다.
- 각 채널은 빨강, 초록, 파랑 정보를 따로 담고 있습니다.

#### 2) 첫 번째 합성곱 블록

- `Conv2d(3, 32, kernel_size=3, padding=1)`
- `BatchNorm2d(32)`
- `ReLU()`
- `Conv2d(32, 32, kernel_size=3, padding=1)`
- `ReLU()`
- `MaxPool2d(2)`

이 구간에서의 변화는 다음과 같습니다.

- 입력: `(3, 32, 32)`
- 첫 번째 합성곱 뒤: `(32, 32, 32)`
- 두 번째 합성곱 뒤: `(32, 32, 32)`
- 풀링 뒤: `(32, 16, 16)`

여기서 중요한 점은 다음과 같습니다.

- 첫 번째 합성곱은 RGB 세 채널을 동시에 보면서 32개의 필터 반응을 만듭니다.
- `padding=1`이라서 `3 x 3` 커널을 써도 가로세로 크기 `32 x 32`가 유지됩니다.
- `BatchNorm`은 값 분포를 정돈해 학습을 좀 더 안정적으로 만들어줍니다.
- `ReLU`는 음수를 잘라내 비선형성을 추가합니다.
- 마지막 `MaxPool2d(2)`는 공간 크기를 절반으로 줄여 중요한 특징만 남기고 계산량을 줄입니다.

이 단계에서는 주로 색 경계, 짧은 선, 코너, 작은 질감 같은 **기초 시각 특징**이 잡히기 시작합니다.

#### 3) 두 번째 합성곱 블록

- `Conv2d(32, 64, kernel_size=3, padding=1)`
- `BatchNorm2d(64)`
- `ReLU()`
- `Conv2d(64, 64, kernel_size=3, padding=1)`
- `ReLU()`
- `MaxPool2d(2)`

shape 변화는 다음과 같습니다.

- 입력: `(32, 16, 16)`
- 첫 번째 합성곱 뒤: `(64, 16, 16)`
- 두 번째 합성곱 뒤: `(64, 16, 16)`
- 풀링 뒤: `(64, 8, 8)`

이제 채널 수가 32에서 64로 늘어납니다. 이는 모델이 더 다양한 특징 조합을 저장할 수 있다는 뜻입니다.

이 단계에서는 단순한 선 하나보다, 예를 들어 다음과 같은 패턴이 더 잘 잡힙니다.

- 둥근 윤곽 일부
- 반복되는 털 질감
- 바퀴처럼 원형에 가까운 패턴
- 날개나 몸통 일부처럼 물체의 부분 구조

#### 4) 세 번째 합성곱 블록

- `Conv2d(64, 128, kernel_size=3, padding=1)`
- `BatchNorm2d(128)`
- `ReLU()`
- `MaxPool2d(2)`

shape 변화는 다음과 같습니다.

- 입력: `(64, 8, 8)`
- 합성곱 뒤: `(128, 8, 8)`
- 풀링 뒤: `(128, 4, 4)`

여기서는 채널 수가 128까지 늘어나고, 공간 크기는 `4 x 4`까지 줄어듭니다. 즉, 위치 정보는 더 압축되지만 채널 차원에서는 더 풍부한 의미를 담게 됩니다.

이 시점의 특징 맵은 단순 색 변화보다 더 큰 단위의 패턴, 즉 물체 일부의 조합을 담는 경향이 있습니다.

#### 5) Flatten: 3차원 특징 맵을 1차원 벡터로 변환

- 입력 shape: `(128, 4, 4)`
- 변환 후 shape: `128 x 4 x 4 = 2048`

합성곱 층은 이미지 구조를 유지하지만, 마지막 분류기는 벡터 입력을 받습니다. 그래서 `Flatten()`으로 3차원 특징 맵을 길이 2048짜리 벡터로 펼칩니다.

#### 6) 완전연결 분류기

- `Linear(2048, 256)`
- `ReLU()`
- `Dropout(0.3)`
- `Linear(256, 10)`

이 구간의 역할은 다음과 같습니다.

- `Linear(2048, 256)`은 앞에서 추출한 특징들을 종합해서 더 압축된 표현으로 만듭니다.
- `ReLU`는 여기서도 비선형 조합을 가능하게 합니다.
- `Dropout(0.3)`은 학습 중 일부 뉴런을 무작위로 끄면서 과적합을 줄입니다.
- 마지막 `Linear(256, 10)`은 각 클래스에 대한 점수(logit) 10개를 출력합니다.

#### 7) 최종 출력

- 출력 shape: `(10,)`
- 각 값은 `airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck` 중 하나에 대한 점수입니다.
- 가장 큰 값을 가진 인덱스가 최종 예측 클래스가 됩니다.

정리하면 이 CNN의 흐름은 다음과 같습니다.

- 앞쪽 합성곱 층: 색상, 경계, 질감 같은 작은 특징 추출
- 중간 합성곱 층: 여러 작은 특징을 조합해 물체 일부 패턴 형성
- 마지막 분류기: 추출된 특징을 종합해 10개 클래스 중 하나로 결정


In [ ]:
class ColorCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model = ColorCNN().to(device)
print(model)
print('Trainable parameters:', f"{count_parameters(model):,}")


### 4-3-3. 코드로 SVG 아키텍처 다이어그램 생성하기

Jupyter Notebook에서는 단순히 텍스트로 모델을 출력하는 것뿐 아니라, 코드로 `SVG` 다이어그램을 생성해 더 선명하게 구조를 표현할 수도 있습니다.

아래 예제는 `ColorCNN`의 레이어를 자동으로 읽어서, 블록 단위의 세로형 구조도를 `SVG`로 만듭니다. `matplotlib`보다 정렬과 선명도가 안정적이고, 화살표 방향도 명확하게 제어할 수 있습니다.


In [ ]:
from html import escape
from IPython.display import SVG, display

def layer_label(name, layer):
    if isinstance(layer, nn.Conv2d):
        return [name, f'Conv2d {layer.in_channels}->{layer.out_channels}', f'kernel={layer.kernel_size}, padding={layer.padding}']
    if isinstance(layer, nn.BatchNorm2d):
        return [name, f'BatchNorm2d {layer.num_features}']
    if isinstance(layer, nn.ReLU):
        return [name, 'ReLU activation']
    if isinstance(layer, nn.MaxPool2d):
        return [name, f'MaxPool2d kernel={layer.kernel_size}', f'stride={layer.stride}']
    if isinstance(layer, nn.Flatten):
        return [name, 'Flatten feature map']
    if isinstance(layer, nn.Linear):
        return [name, f'Linear {layer.in_features}->{layer.out_features}']
    if isinstance(layer, nn.Dropout):
        return [name, f'Dropout p={layer.p}']
    return [name, layer.__class__.__name__]

def layer_color(layer):
    mapping = {
        nn.Conv2d: '#FFEDD5',
        nn.BatchNorm2d: '#DBEAFE',
        nn.ReLU: '#F3E8FF',
        nn.MaxPool2d: '#FEF3C7',
        nn.Flatten: '#DCFCE7',
        nn.Linear: '#FEE2E2',
        nn.Dropout: '#E5E7EB'
    }
    return mapping.get(type(layer), '#FFFFFF')

layers = []
for name, layer in model.named_modules():
    if not name or isinstance(layer, (ColorCNN, nn.Sequential)):
        continue
    layers.append((name, layer))

box_x = 210
box_w = 520
box_h = 74
gap = 22
top = 110
height = top + len(layers) * (box_h + gap) + 80
width = 940

svg = [
    f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
    "<defs>",
    "  <filter id='shadow2' x='-20%' y='-20%' width='140%' height='140%'>",
    "    <feDropShadow dx='0' dy='5' stdDeviation='8' flood-color='#0F172A' flood-opacity='0.10' />",
    "  </filter>",
    "  <marker id='arrow2' markerWidth='10' markerHeight='10' refX='9' refY='5' orient='auto'>",
    "    <path d='M 0 0 L 10 5 L 0 10 z' fill='#64748B' />",
    "  </marker>",
    "</defs>",
    "<rect width='100%' height='100%' rx='24' fill='#FFFFFF' />",
    "<text x='470' y='42' text-anchor='middle' font-size='25' font-weight='700' fill='#111827' font-family='Segoe UI, Arial, sans-serif'>ColorCNN Auto-Generated SVG Diagram</text>",
    "<text x='470' y='68' text-anchor='middle' font-size='13' fill='#475569' font-family='Segoe UI, Arial, sans-serif'>Each box is derived directly from the PyTorch model definition</text>",
    "<text x='470' y='92' text-anchor='middle' font-size='14' font-weight='600' fill='#0F172A' font-family='Segoe UI, Arial, sans-serif'>Input: (3, 32, 32)</text>"
]

for i, (name, layer) in enumerate(layers):
    y = top + i * (box_h + gap)
    fill = layer_color(layer)
    lines = layer_label(name, layer)
    svg.append(f"<rect x='{box_x}' y='{y}' width='{box_w}' height='{box_h}' rx='18' fill='{fill}' stroke='#CBD5E1' stroke-width='1.4' filter='url(#shadow2)' />")
    for j, line in enumerate(lines):
        font_size = 15 if j == 0 else 13
        font_weight = '700' if j == 0 else '500'
        color = '#111827' if j == 0 else '#334155'
        text_y = y + 24 + j * 18
        svg.append(
            f"<text x='{box_x + box_w / 2}' y='{text_y}' text-anchor='middle' font-size='{font_size}' font-weight='{font_weight}' fill='{color}' font-family='Segoe UI, Arial, sans-serif'>{escape(line)}</text>"
        )

    if i < len(layers) - 1:
        y1 = y + box_h
        y2 = y + box_h + gap - 6
        x = box_x + box_w / 2
        svg.append(f"<line x1='{x}' y1='{y1}' x2='{x}' y2='{y2}' stroke='#64748B' stroke-width='2.5' marker-end='url(#arrow2)' />")

svg.append(f"<text x='470' y='{height - 24}' text-anchor='middle' font-size='14' font-weight='600' fill='#0F172A' font-family='Segoe UI, Arial, sans-serif'>Output: logits for 10 CIFAR-10 classes</text>")
svg.append("</svg>")
display(SVG(''.join(svg)))


## 4-4. 모델 학습

아래 학습 코드는 매 epoch마다 train / validation 성능을 함께 확인합니다. CPU만 사용할 때는 시간이 꽤 걸릴 수 있으니, 처음에는 `epochs = 3` 정도로 시작해도 충분합니다.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

def train_model(model, train_loader, val_loader, epochs=5):
    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    history = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        val_loss, val_acc = evaluate(model, val_loader)

        history.append((train_loss, train_acc, val_loss, val_acc))
        print(f'Epoch {epoch + 1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return history

history = train_model(model, train_loader, val_loader, epochs=5)
test_loss, test_acc = evaluate(model, test_loader)
print('Test Loss:', round(test_loss, 4), '| Test Acc:', round(test_acc, 4))


In [ ]:
epochs = range(1, len(history) + 1)
train_losses = [x[0] for x in history]
train_accs = [x[1] for x in history]
val_losses = [x[2] for x in history]
val_accs = [x[3] for x in history]

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, marker='o', label='Train Loss')
plt.plot(epochs, val_losses, marker='o', label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, train_accs, marker='o', label='Train Acc')
plt.plot(epochs, val_accs, marker='o', label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve')
plt.legend()

plt.tight_layout()
plt.show()


컬러 이미지 데이터셋에서는 MNIST보다 정확도가 낮게 시작하는 것이 자연스럽습니다. 문제 자체가 더 어렵기 때문입니다.

그래도 CNN은 여러 층을 거치며 색상, 윤곽선, 질감, 물체 일부 패턴을 점진적으로 조합해서 최종 클래스를 구분합니다.


## 4-5. 예측 결과 확인


In [ ]:
model.eval()
images, labels = next(iter(test_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)
    preds = outputs.argmax(dim=1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    image = denormalize(images[i]).permute(1, 2, 0)
    true_label = classes[labels[i].item()]
    pred_label = classes[preds[i].item()]

    ax.imshow(image)
    ax.set_title(f'T: {true_label}\nP: {pred_label}')
    ax.axis('off')

plt.suptitle('Prediction examples on CIFAR-10')
plt.tight_layout()
plt.show()


예측 결과를 보면, 비슷한 동물끼리 헷갈리거나 자동차와 트럭처럼 모양이 비슷한 클래스를 혼동하는 경우가 있습니다. 이 역시 실제 컬러 이미지 분류가 MNIST보다 훨씬 어렵다는 점을 보여줍니다.


## 4-6. 첫 번째 합성곱 필터 시각화

첫 번째 합성곱 층의 가중치는 `(출력 채널 수, 입력 채널 수, 커널 높이, 커널 너비)` 형태입니다. 여기서는 각 필터가 RGB 채널을 어떻게 바라보는지 간단히 시각화합니다.


In [ ]:
filters = model.features[0].weight.detach().cpu()
num_filters = min(8, filters.size(0))

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    if i >= num_filters:
        ax.axis('off')
        continue

    filt = filters[i]
    filt = filt - filt.min()
    filt = filt / (filt.max() + 1e-8)
    filt = filt.permute(1, 2, 0)

    ax.imshow(filt)
    ax.set_title(f'Filter {i}')
    ax.axis('off')

plt.suptitle('First convolution filters')
plt.tight_layout()
plt.show()


필터 시각화를 보면, 어떤 필터는 특정 색 대비에 민감하고 어떤 필터는 경계선이나 질감에 민감한 형태로 학습됩니다.

즉, 컬러 이미지에서 CNN은 다음과 같이 동작합니다.

- 초기 층: 색상, 경계, 방향성 같은 단순 특징 추출
- 중간 층: 귀, 바퀴, 날개처럼 물체 일부 패턴 조합
- 마지막 층: 전체 물체를 클래스 단위로 구분

이제 CNN이 단순한 흑백 숫자뿐 아니라, 더 복잡한 실제 컬러 이미지에서도 특징을 계층적으로 추출하며 동작한다는 점을 확인할 수 있습니다.

다음 단계에서는 ResNet 같은 더 깊은 CNN 구조를 사용하거나, 전이학습을 통해 더 큰 실제 이미지 문제로 확장해볼 수 있습니다.
